In [ ]:
def register(
    dispatch_mode=Dispatch.ALL_TO_ALL,
    execute_mode=Execute.ALL,
    blocking=True,
    materialize_futures=True,
):
    """Register a function with distributed execution configuration. 注册一个函数，并配置分布式执行。

    This decorator registers a function with specific dispatch and execution modes
    for distributed computation. It handles both synchronous and asynchronous
    functions, and optionally materializes futures before execution.

    Args:
        dispatch_mode:
            Dispatch mode for computation distribution. Default: Dispatch.ALL_TO_ALL.
        execute_mode:
            Execute mode for computation distribution. Default: Execute.ALL.
        blocking:
            Whether the execution should be blocking. Defaults to True.
        materialize_futures:
            Whether to materialize the data before dispatching. Defaults to True.

    Returns:
        A decorator that wraps the original function with distributed execution
        configuration.
    """
    _check_dispatch_mode(dispatch_mode=dispatch_mode)
    _check_execute_mode(execute_mode=execute_mode)

    """
    包装函数：处理异步与数据“变现”

    这里定义了真正的包装逻辑，它做了两件事：
    1. 数据“变现” (_materialize_futures)：
        - 在分布式系统中，参数可能是一个“承诺”（Future/Promise），代表一个还没计算完的结果。
        - 如果 materialize_futures=True，装饰器会在执行函数前，强制等待这些 Future 完成，把数据真正取出来（即“变现”），确保函数内部拿到的是实实在在的数据，而不是句柄。
    2. 兼容异步：
        - 代码检测了原函数是否是 async 的（inspect.iscoroutinefunction），
          并分别生成了同步版 inner 和异步版 async_inner 包装器，保证框架能同时支持同步和异步代码。


    """

    def decorator(func):
        @wraps(func)
        def inner(*args, **kwargs):
            '''
            强制等待这些 Future 完成，把数据真正取出来
            '''
            if materialize_futures:
                args, kwargs = _materialize_futures(*args, **kwargs)
            return func(*args, **kwargs)

        @wraps(func)
        async def async_inner(*args, **kwargs):
            '''
            强制等待这些 Future 完成，把数据真正取出来
            '''
            if materialize_futures:
                args, kwargs = _materialize_futures(*args, **kwargs)
            return await func(*args, **kwargs)

        """
        注入“魔法”：元数据标记

        这是整个装饰器最精髓的部分：
        - MAGIC_ATTR：这是一个特殊的字符串常量（如 "attrs_3141562937"）。
        - setattr(...)：装饰器并没有直接执行分布式逻辑，而是把你配置的参数（分发模式、执行模式等）打包成一个字典，像贴标签一样贴在了函数对象上。
        
        为什么要这么做？
        - 这是一种元编程技巧。VeRL 的框架层（如 RayWorkerGroup）在初始化时，会扫描所有 Worker 类的方法。
          一旦发现某个方法身上有这个 MAGIC_ATTR 标签，框架就知道：“哦，这个方法需要分布式处理”，
          然后框架会根据标签里的配置，动态生成相应的分发和收集代码。
        """
        wrapper = async_inner if inspect.iscoroutinefunction(func) else inner
        attrs = {
            "dispatch_mode": dispatch_mode,
            "execute_mode": execute_mode,
            "blocking": blocking,
        }
        setattr(wrapper, MAGIC_ATTR, attrs)
        return wrapper

    return decorator


## 它的核心哲学是：业务逻辑与分布式逻辑解耦。

- <font color='red'>开发者只需关注函数内部的计算逻辑（“做什么”），而通过装饰器参数告诉框架如何处理数据的分发和同步（“怎么做”）。</font>

这是装饰器的工厂函数，它允许你为被装饰的方法配置“元数据”：
- dispatch_mode: <font color='red'>决定数据怎么分。</font>例如是广播给所有人（ONE_TO_ALL），还是切分给不同人（ALL_TO_ALL）。
- execute_mode: <font color='red'>决定谁去执行。</font>是所有 Worker 都跑一遍（ALL），还是只有主节点跑（RANK_ZERO）。
- blocking: 决定调用后是否等待结果。
- materialize_futures: 这是一个高级优化选项，用于处理异步对象（Ray Future）。


这个 register 函数通过闭包和元编程，实现了以下流程：
1.接收配置：用户指定怎么分发数据。
2.预处理数据：在执行前自动把异步 Future 转为实体数据。
3.打标签：把配置信息绑定在函数属性上，供框架后续读取和调度。

<font color='red'>这使得 VeRL 的代码看起来非常像单机代码，但背后却拥有强大的分布式调度能力。</font>

## _check_dispatch_mode()

In [ ]:
# Global registry for dispatch mode.
DISPATCH_MODE_FN_REGISTRY = {
    Dispatch.ONE_TO_ALL: {
        "dispatch_fn": dispatch_one_to_all,
        "collect_fn": collect_all_to_all,
    },
    Dispatch.ALL_TO_ALL: {
        "dispatch_fn": dispatch_all_to_all,
        "collect_fn": collect_all_to_all,
    },
    Dispatch.DP_COMPUTE: {
        "dispatch_fn": dispatch_dp_compute,
        "collect_fn": collect_dp_compute,
    },
    Dispatch.DP_COMPUTE_PROTO: {
        "dispatch_fn": dispatch_dp_compute_data_proto,
        "collect_fn": collect_dp_compute_data_proto,
    },
    Dispatch.DP_COMPUTE_PROTO_WITH_FUNC: {
        "dispatch_fn": dispatch_dp_compute_data_proto_with_func,
        "collect_fn": collect_dp_compute_data_proto,
    },
    Dispatch.DP_COMPUTE_METRIC: {
        "dispatch_fn": dispatch_dp_compute_data_proto,
        "collect_fn": collect_dp_compute,
    },
    Dispatch.DIRECT_ROLLOUT_METHOD: {
        "dispatch_fn": dummy_direct_rollout_call,
        "collect_fn": dummy_direct_rollout_call,
    },
}

In [ ]:
def _check_dispatch_mode(dispatch_mode):
    '''
    它要求 dispatch_mode 必须是 Dispatch 类的实例或者是一个字典（dict）。
    
    强类型对象：直接传入封装好的 Dispatch 类对象。
    灵活的字典：传入一个包含必要回调函数（dispatch_fn 和 collect_fn）的字典，方便快速配置。
    '''
    assert isinstance(
        dispatch_mode, Dispatch | dict
    ), f"dispatch_mode must be a Dispatch or a Dict. Got {dispatch_mode}"
    
    if isinstance(dispatch_mode, dict):
        '''
        定义了一个必须包含的键列表 necessary_keys，
        其中包含 "dispatch_fn"（分发函数）和 "collect_fn"（收集/聚合函数）
        
        如果字典中缺少了这两个键中的任意一个，程序同样会抛出 AssertionError，明确指出缺失的键名。
        '''
        necessary_keys = ["dispatch_fn", "collect_fn"]
        for key in necessary_keys:
            assert (
                key in dispatch_mode
            ), f"key {key} should be in dispatch_mode if it is a dictionary"


## _check_execute_mode()

In [ ]:
def _check_execute_mode(execute_mode):
    assert isinstance(
        execute_mode, Execute
    ), f"execute_mode must be a Execute. Got {execute_mode}"
